In [15]:
import pandas as pd
import seaborn as sns
import urllib
import sqlalchemy
import psycopg2
import sys
import gradio as gr

In [16]:
from sqlalchemy import create_engine, text

In [17]:
db_params = {
    'dbname': 'guest',
    'user': 'guest',
    'password': '12345',
    'host': 'localhost',
    'port': '5432'
}

engine = create_engine(f'postgresql+psycopg2://{db_params['user']}:{db_params['password']}@{db_params["host"]}:{db_params['port']}/{db_params['dbname']}')

In [29]:
# Функция для получения всех уникальных предметных областей
def get_all_subject_areas():
    query = text("""
    SELECT DISTINCT subjareas
    FROM full_database_normalised
    WHERE subjareas IS NOT NULL AND subjareas != ''
    ORDER BY subjareas
    """)
    
    try:
        with engine.connect() as conn:
            df = pd.read_sql(query, conn)
            
            #Список всех уникальных предметных областей
            all_areas = []
            for areas in df['subjareas']:
                if pd.notna(areas) and areas:
                    for area in areas.split(','):
                        area = area.strip()
                        if area and area not in all_areas:
                            all_areas.append(area)
            
            return sorted(all_areas)
    
    except Exception as e:
        print(f"Ошибка при получении предметных областей: {e}")
        return []

def update_query_filter_by_subject_areas(base_query, params, subject_areas=None):
    if subject_areas and len(subject_areas) > 0:
        area_conditions = []
        for i, area in enumerate(subject_areas):
            area_conditions.append(f"subjareas LIKE :area{i}")
            params[f"area{i}"] = f'%{area}%'

        if area_conditions:
            base_query += " AND (" + " OR ".join(area_conditions) + ")"
        
        return base_query, params
    return base_query, params

# Функция для поиска статей по фамилии автора и предметным областям
def search_by_author(author_surname, subject_areas=None):
    
    if not author_surname:
        return "Введите фамилию автора", pd.DataFrame()
    
    #Базовый запрос с поиском в main_author и secondary_authors
    base_query = """
    SELECT 
        main_author, secondary_authors, title, doi, subjareas, keywords
    FROM 
        full_database_normalised
    WHERE 
        (LOWER(main_author) LIKE LOWER(:search_term)
        OR LOWER(secondary_authors) LIKE LOWER(:search_term))
    """
    
    # Создаем словарь для хранения параметров запроса
    params = {"search_term": f'%{author_surname}%'}
    
    #Фильтрация по предметным областям
    base_query, params = update_query_filter_by_subject_areas(base_query, params, subject_areas)
    
    base_query += " ORDER BY main_author"
    
    query = text(base_query)
    
    try:
        with engine.connect() as conn:
            df = pd.read_sql(query, conn, params=params)
            
            if df.empty:
                message = f"По запросу '{author_surname}'"
                if subject_areas and len(subject_areas) > 0:
                    message += f" в выбранных предметных областях"
                message += " ничего не найдено"
                return message, pd.DataFrame()
            
            # Сообщение с количеством найденных статей
            message = f"Найдено {len(df)} статей для автора '{author_surname}'"
            if subject_areas and len(subject_areas) > 0:
                message += f" в выбранных предметных областях ({', '.join(subject_areas)})"
        
            return message, df
    
    except Exception as e:
        return f"Ошибка: {str(e)}", pd.DataFrame()

# Функция для получения соавторов по фамилии автора и предметным областям
def get_coauthors(author_surname, subject_areas=None):
    if not author_surname:
        return "Введите фамилию автора", pd.DataFrame()
    
    # Базовый запрос для получения всех статей, где искомый автор является main_author или secondary_authors
    base_query = """
    SELECT 
        main_author, secondary_authors, title, doi, subjareas
    FROM 
        full_database_normalised
    WHERE 
        (LOWER(main_author) LIKE LOWER(:search_term)
        OR LOWER(secondary_authors) LIKE LOWER(:search_term))
    """
    
    # Создаем словарь для хранения параметров запроса
    params = {"search_term": f'%{author_surname}%'}
    
    #Фильтрация по предметным областям
    base_query, params = update_query_filter_by_subject_areas(base_query, params, subject_areas)
    
    query = text(base_query)
    
    try:
        with engine.connect() as conn:
            df = pd.read_sql(query, conn, params=params)
            
            if df.empty:
                message = f"По запросу '{author_surname}'"
                if subject_areas and len(subject_areas) > 0:
                    message += f" в выбранных предметных областях"
                message += " ничего не найдено"
                return message, pd.DataFrame()
            
            #Список для соавторов
            coauthors_data = []
            
            #Словарь для информации о совместных статьях
            coauthors_papers = {}
            
            #Идем по статьям
            for _, row in df.iterrows():
                main_author = row['main_author']
                secondary_authors = row['secondary_authors'] if pd.notna(row['secondary_authors']) else ""
                title = row['title']
                doi = row['doi']
                
                #Добавляем всех соавторов в список
                all_authors = []
                if main_author and author_surname.lower() not in main_author.lower():
                    all_authors.append(main_author)
                
                if secondary_authors:
                    for coauthor in secondary_authors.split(','):
                        coauthor = coauthor.strip()
                        if coauthor and author_surname.lower() not in coauthor.lower():
                            all_authors.append(coauthor)
                
                #Добавляем инфу о соавторах
                for coauthor in all_authors:
                    paper_info = {
                        'title': title,
                        'doi': doi
                    }
                    
                    if coauthor not in coauthors_papers:
                        coauthors_papers[coauthor] = []
                    
                    #по автору добавляем информацию о статье
                    coauthors_papers[coauthor].append(paper_info)
                    
                    coauthors_data.append({
                        'coauthor': coauthor,
                        'title': title,
                        'doi': doi
                    })
            
            # Создаем датафрейм с соавторами
            coauthors_df = pd.DataFrame(coauthors_data)
            
            if coauthors_df.empty:
                message = f"Для автора '{author_surname}'"
                if subject_areas and len(subject_areas) > 0:
                    message += f" в выбранных предметных областях"
                message += " не найдено соавторов"
                return message, pd.DataFrame()
            
            #Группируем соавторов и считаем количество совместных работ
            coauthors_count = coauthors_df.groupby('coauthor').size().reset_index(name='joint_papers')
            coauthors_count = coauthors_count.sort_values('joint_papers', ascending=False)
            
            # Добавляем информацию о совместных статьях
            coauthors_count['papers'] = coauthors_count['coauthor'].apply(
                lambda x: '; '.join([f"{i+1}. {p['title']} (DOI: {p['doi']})" for i, p in enumerate(coauthors_papers[x])])
            )
            
            # Сообщение о количестве найденных соавторов
            message = f"Найдено {len(coauthors_count)} соавторов для '{author_surname}'"
            if subject_areas and len(subject_areas) > 0:
                message += f" в выбранных предметных областях ({', '.join(subject_areas)})"
            
            return message, coauthors_count
    
    except Exception as e:
        return f"Ошибка: {str(e)}", pd.DataFrame()

# Функция для поиска авторов по ключевым словам и предметным областям
def search_authors_by_keywords(keywords, subject_areas=None):
    if not keywords:
        return "Введите ключевые слова", pd.DataFrame()
    
    #Делаем список из введенных ключевых слов
    keywords_list = [kw.strip().lower() for kw in keywords.split(',')]
    
    params = {}
    
    # Создаем условия для SQL-запроса по ключевым словам
    conditions = []
    for i, kw in enumerate(keywords_list):
        conditions.append(f"LOWER(keywords) LIKE :keyword{i}")
        params[f"keyword{i}"] = f"%{kw}%"
    
    conditions_str = " OR ".join(conditions)
    
    # Базовый запрос для поиска статей по ключевым словам
    base_query = f"""
    SELECT 
        main_author, secondary_authors, title, keywords, doi, subjareas
    FROM 
        full_database_normalised
    WHERE 
        ({conditions_str})
    """
    
    #Фильтр предметных областей
    base_query, params = update_query_filter_by_subject_areas(base_query, params, subject_areas)
    
    query = text(base_query)
    
    try:
        with engine.connect() as conn:
            df = pd.read_sql(query, conn, params=params)
            
            if df.empty:
                keywords_str = ", ".join(keywords_list)
                message = f"По ключевым словам '{keywords_str}'"
                if subject_areas and len(subject_areas) > 0:
                    message += f" в выбранных предметных областях"
                message += " ничего не найдено"
                return message, pd.DataFrame()
            
            # Создаем список для хранения информации об авторах
            authors_data = []
            
            # Словарь для хранения информации о статьях
            authors_papers = {}
            
            #Обрабатываем каждую статью
            for _, row in df.iterrows():
                main_author = row['main_author'] if pd.notna(row['main_author']) else ""
                secondary_authors = row['secondary_authors'] if pd.notna(row['secondary_authors']) else ""
                title = row['title']
                found_keywords = row['keywords']
                doi = row['doi']
                
                # Функция для добавления автора в список данных
                def add_author_to_data(author):
                    paper_info = {
                        'title': title,
                        'keywords': found_keywords,
                        'doi': doi
                    }
                    
                    # Инициализируем список статей для автора, если его еще нет
                    if author not in authors_papers:
                        authors_papers[author] = []
                    
                    # Добавляем информацию о статье
                    authors_papers[author].append(paper_info)
                    
                    authors_data.append({
                        'author': author,
                        'title': title,
                        'keywords': found_keywords,
                        'doi': doi
                    })
                
                #Добавляем главного автора в список
                if main_author:
                    add_author_to_data(main_author)
                
                #И вторичных авторов 
                if secondary_authors:
                    for author in secondary_authors.split(','):
                        author = author.strip()
                        if author:
                            add_author_to_data(author)
            
            authors_df = pd.DataFrame(authors_data)
            
            #Группируем и считаем количество статей по нашей теме
            authors_count = authors_df.groupby('author').size().reset_index(name='papers_count')
            authors_count = authors_count.sort_values('papers_count', ascending=False)
            
            # Добавляем информацию о статьях
            authors_count['papers'] = authors_count['author'].apply(
                lambda x: '; '.join([f"{i+1}. {p['title']} (DOI: {p['doi']})" for i, p in enumerate(authors_papers[x])])
            )
            
            # Количество найденных авторов
            keywords_str = ", ".join(keywords_list)
            message = f"Найдено {len(authors_count)} авторов по ключевым словам '{keywords_str}'"
            if subject_areas and len(subject_areas) > 0:
                message += f" в выбранных предметных областях ({', '.join(subject_areas)})"
            
            return message, authors_count
    
    except Exception as e:
        return f"Ошибка: {str(e)}", pd.DataFrame()
    
# Тут Gradio интерфейс
with gr.Blocks(title="Научные статьи - интерфейс") as demo:
    gr.Markdown("# Интерфейс для работы с базой данных статей сотрудников НИУ ВШЭ")
    
    #список всех предметных областей при инициализации
    subject_areas_list = get_all_subject_areas()
    
    with gr.Tab("Поиск статей по автору"):
        with gr.Row():
            author_input = gr.Textbox(label="Введите фамилию автора")
            subject_areas_dropdown1 = gr.Dropdown(
                choices=subject_areas_list,
                label="Выберите предметные области (опционально)",
                multiselect=True
            )
            search_button = gr.Button("Найти статьи")
        
        result_message1 = gr.Textbox(label="Результат поиска", lines=2)
        author_output = gr.Dataframe(label="Найденные статьи", wrap=True)
        
        with gr.Row():
            copy_button1 = gr.Button("Скопировать данные")
        
        # Генерируем текст,чтобы скопировать его
        def prepare_copy_text(df):
            if df.empty:
                return "Нет данных для копирования"
            
            #DataFrame в текст
            text = df.to_csv(index=False, sep="\t")
            return text
        
        copy_text1 = gr.Textbox(visible=True, label="Скопируйте эти данные (Ctrl+A, затем Ctrl+C)")
        
        search_button.click(fn=search_by_author, inputs=[author_input, subject_areas_dropdown1], outputs=[result_message1, author_output])
        copy_button1.click(fn=prepare_copy_text, inputs=[author_output], outputs=[copy_text1])
    
    with gr.Tab("Поиск соавторов"):
        with gr.Row():
            coauthor_input = gr.Textbox(label="Введите фамилию автора")
            subject_areas_dropdown2 = gr.Dropdown(
                choices=subject_areas_list,
                label="Выберите предметные области (опционально)",
                multiselect=True
            )
            coauthors_button = gr.Button("Найти соавторов")
        
        result_message2 = gr.Textbox(label="Результат поиска", lines=2)
        coauthors_output = gr.Dataframe(label="Найденные соавторы", wrap=True)
        
        with gr.Row():
            copy_button2 = gr.Button("Скопировать данные")
        
        copy_text2 = gr.Textbox(visible=True, label="Скопируйте эти данные (Ctrl+A, затем Ctrl+C)")
        
        coauthors_button.click(fn=get_coauthors, inputs=[coauthor_input, subject_areas_dropdown2], outputs=[result_message2, coauthors_output])
        copy_button2.click(fn=prepare_copy_text, inputs=[coauthors_output], outputs=[copy_text2])
    
    with gr.Tab("Поиск авторов по ключевым словам"):
        with gr.Row():
            keywords_input = gr.Textbox(label="Введите ключевые слова (через запятую)")
            subject_areas_dropdown3 = gr.Dropdown(
                choices=subject_areas_list,
                label="Выберите предметные области (опционально)",
                multiselect=True
            )
            keywords_button = gr.Button("Найти авторов")
        
        result_message3 = gr.Textbox(label="Результат поиска", lines=2)
        keywords_output = gr.Dataframe(label="Найденные авторы", wrap=True)
        
        with gr.Row():
            copy_button3 = gr.Button("Скопировать данные")
        
        copy_text3 = gr.Textbox(visible=True, label="Скопируйте эти данные (Ctrl+A, затем Ctrl+C)")
        
        keywords_button.click(fn=search_authors_by_keywords, inputs=[keywords_input, subject_areas_dropdown3], outputs=[result_message3, keywords_output])
        copy_button3.click(fn=prepare_copy_text, inputs=[keywords_output], outputs=[copy_text3])

if __name__ == "__main__":
    demo.launch(share=True)

* Running on local URL:  http://127.0.0.1:7922
* Running on public URL: https://cbf8aaa64e8f87fa35.gradio.live

This share link expires in 72 hours. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
